# Ray + Modal: standalone multi-node pattern

This tutorial shows how to stand up a Ray cluster across Modal
containers and submit jobs to it via Ray's `JobSubmissionClient`. The
SLIME framework package (`modal_training_gym.frameworks.slime`) builds
on exactly this pattern — here we use the `ModalRayCluster` helper
directly, so you can see what's going on without any framework-specific
sugar.

Useful when you want a Ray cluster on Modal but your training code
isn't one of the frameworks we've wrapped.

## Design

- **Bootstrap** — on every clustered container, call
  `ModalRayCluster().start(n_nodes=...)`. Rank 0 starts the Ray head
  and waits for all workers to join; other ranks start Ray workers
  that register with the head.
- **Job submission** — on the head only, use the cluster's
  `JobSubmissionClient` to submit a Python entrypoint and stream its
  logs.
- **Dashboard** — open the Ray dashboard via `modal.forward` so you
  can watch the job from your browser.

In [ ]:
! pip install -q git+https://github.com/modal-projects/training-gym.git@joy/initial-setup

In [ ]:
import modal

from modal_training_gym.common.ray_cluster import ModalRayCluster

N_NODES = 2

# Any image with Ray installed works. We use the SLIME nightly image
# for parity with the framework-backed SLIME tutorial.
image = (
    modal.Image.from_registry("slimerl/slime:nightly-dev-20260329a")
    .entrypoint([])
    .add_local_python_source("modal_training_gym", copy=True)
)

app = modal.App("ray-standalone", image=image)

## Clustered Ray + job submission

One `@modal.experimental.clustered`-decorated function does it all:
bootstrap Ray on every rank, then (on head) submit a Ray job and tail
its logs. Other ranks sit in `wait_forever()` so the cluster stays up
until the head finishes.

In [ ]:
async def run_ray_job(entrypoint: str = "python -c 'import ray; ray.init(); print(ray.cluster_resources())'"):
    """Spin up Ray across the cluster, submit `entrypoint`, and tail its logs.

    The default entrypoint just prints Ray's view of cluster resources —
    swap in any shell command (e.g. `python3 my_train.py --flags …`) to
    run a real job.
    """
    cluster = ModalRayCluster()
    cluster.start(n_nodes=N_NODES)

    if not cluster.is_head:
        await cluster.wait_forever()
        return

    async with cluster.forward_dashboard() as tunnel:
        print(f"Ray dashboard: {tunnel.url}")
        status = await cluster.submit_and_tail(entrypoint)
        print(f"Final status: {status}")

In [ ]:
with app.run():
    run_ray_job.remote()

## Customization

To run a real training workload:

- Override `image` above with the framework's expected environment.
- Add volumes for datasets / checkpoints / HF cache as needed.
- Pass the training command as `entrypoint`; `JobSubmissionClient`
  runs it on the head and streams logs over the Ray protocol.

If you find yourself needing framework-specific plumbing (data prep,
checkpoint conversion, etc.), the `slime` / `verl` framework packages
already wrap this same pattern plus their respective training CLIs.